# SpeakSafe – Model Training Notebook

This notebook walks through the full training pipeline for the SpeakSafe AI voice classifier.

## Steps
1. Load and explore the dataset
2. Extract acoustic features
3. Feature analysis & visualisation
4. Train GradientBoosting classifier
5. Evaluate + calibrate
6. Export model

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import librosa
import librosa.display
import pandas as pd
from pathlib import Path
import sys
sys.path.insert(0, '../backend')
from audio_features import extract_features
from model import classify_audio

print('Dependencies loaded ✓')

## 1. Dataset Overview

In [ ]:
DATA_DIR = Path('../data')
human_files = list((DATA_DIR / 'human').glob('*.wav')) + list((DATA_DIR / 'human').glob('*.mp3'))
ai_files    = list((DATA_DIR / 'ai').glob('*.wav'))    + list((DATA_DIR / 'ai').glob('*.mp3'))

print(f'Human samples : {len(human_files)}')
print(f'AI samples    : {len(ai_files)}')
print(f'Total         : {len(human_files) + len(ai_files)}')

## 2. Feature Extraction

In [ ]:
rows = []
for fp in human_files[:200]:
    try:
        f = extract_features(str(fp))
        f['label'] = 0
        rows.append(f)
    except Exception as e:
        print(f'Skip {fp.name}: {e}')

for fp in ai_files[:200]:
    try:
        f = extract_features(str(fp))
        f['label'] = 1
        rows.append(f)
    except Exception as e:
        print(f'Skip {fp.name}: {e}')

df = pd.DataFrame(rows)
print(df.shape)
df.head()

## 3. Feature Distribution Analysis

In [ ]:
key_features = [
    'mfcc_variance_regularity',
    'spectral_centroid_stability',
    'zcr_regularity',
    'rms_uniformity',
    'mel_banding_score',
    'harmonic_dominance',
]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Feature Distributions: Human vs AI', fontsize=14, fontweight='bold')

for ax, feat in zip(axes.flat, key_features):
    human_vals = df[df['label']==0][feat].dropna()
    ai_vals    = df[df['label']==1][feat].dropna()
    ax.hist(human_vals, bins=30, alpha=0.6, color='#4dffb8', label='Human', density=True)
    ax.hist(ai_vals,    bins=30, alpha=0.6, color='#ff4d6d', label='AI',    density=True)
    ax.set_title(feat.replace('_',' ').title())
    ax.legend(fontsize=8)
    ax.set_xlabel('Value')
    ax.set_ylabel('Density')

plt.tight_layout()
plt.savefig('../docs/feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Spectrogram Comparison

In [ ]:
if human_files and ai_files:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for ax, fp, title, cmap in [
        (axes[0], human_files[0], 'Human Speech', 'magma'),
        (axes[1], ai_files[0],    'AI Generated', 'inferno'),
    ]:
        y, sr = librosa.load(str(fp), sr=16000, duration=3.0)
        S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
        S_db = librosa.power_to_db(S, ref=np.max)
        img = librosa.display.specshow(S_db, sr=sr, hop_length=160,
                                       x_axis='time', y_axis='mel', ax=ax, cmap=cmap)
        fig.colorbar(img, ax=ax, format='%+2.0f dB')
        ax.set_title(title, fontweight='bold')

    plt.tight_layout()
    plt.savefig('../docs/spectrogram_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()

## 5. Train & Evaluate

In [ ]:
# Run the full training pipeline
import subprocess
result = subprocess.run(
    ['python', '../scripts/train.py', '--data-dir', '../data/', '--out', '../models/speaksafe_v1.pkl'],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)